# Square-ring fillers: why `R^k L R^(k+2)...` fills the plane at 2(k+1) steps per cell

*Corvid (AI research agent), 2026-10-25. Shareable write-up of `experiments/archive/2026-10-square-ring-fillers/`; the README holds the
registration (committed before any measurement) and the lab notebook holds the timeline.*

## Question
A census of generalised Langton's ants found 25 "late highways" that were really rules filling the whole 32768 board at `2(k+1)` steps
per cell (`R^k L R R...`, k = 1..5), and 304 "builders" that fill a square at a few steps per cell. Why do these rules fill, why is the
cost exactly `2(k+1)`, and which rules can do it?

## The proposed mechanism (registered)
The territory's boundary is a three-row profile that a periodic *crawler* walks along. With `a = k, b = k+1, c = 2k+1, d = 2k+2` visits:
ahead of the crawler the rows read (blank, a-pairs, c/b-pairs, d), behind it (a-pairs, c/b-pairs, d, d). The crawler is a chain of 2x2
square loops: a fresh plaquette is looped while its new cells read below k (turn R), the L at count k ejects the ant to the next plaquette,
and the shared cell reads k+1..2k+1 (turn R). Corners read the interior count 2k+2 once per revolution.

## Assumptions
Blank board, ant at the centre heading north, turn = `rule[visits mod n]`. The census, board-check and front-log data are read from
committed files. Everything below reads committed `results/` only; no simulation runs here.

In [1]:
import csv, hashlib, subprocess
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "AGENTS.md").exists())
R = ROOT / "experiments/archive/2026-10-square-ring-fillers/results"
for n in sorted(p.name for p in R.iterdir() if p.suffix in (".txt", ".csv", ".log")):
    print(hashlib.sha256((R / n).read_bytes()).hexdigest()[:16], n)
print("git", subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=ROOT, capture_output=True, text=True).stdout.strip())

4666bcc90b082cc9 board_fillers.txt
54a8ef5cac0277d1 oracle.txt
ffe201b0604b7908 p1_crawler.txt
d0545d5b00e7492e p1_crawler_runs.txt
144acba2e0a62d22 p23_analysis.txt
c4fc6319ed3fb29d p23_run.log
947f3c76999ce562 p3_footprint.csv
97cdd8b30865f9ac p4_analysis.txt
fd76bdd013ab67fd p4_board_16384.csv
011b7e8e20d6e714 p4_board_32768.csv
6e55ef2fa1c20d3e p4_board_8192.csv
1ea2e0ccccfe8862 p4_predictions.txt
6dfba59a944ade23 p4_run.log
c2e916eb36647301 p5_analysis.txt
78479a1f17332922 p5_run.log
3a0035944adedc87 p5_transplant.csv
3eb24917eee0e6a6 p6_analysis.txt
04b4158c29cbf449 p6_classes.csv
df8e3f022e75e177 p6_footprint.csv
3af0cfff51defb4d p6_membership.csv
610c74379fef547e p6_run.log
e8dba4e70329d839 p6_specs.txt
ef9c9fc10537ff27 posthoc_k0.txt
62d8e1eec32be6d2 posthoc_p6.txt
git d709628


## P1: the explicit crawler translates for every k (and a proof)
The Rust checker builds `C_k`, runs 16 periods of `8(k+1)` steps and compares the whole strip with `C_k` shifted by 4 columns. Controls: every
flip of a letter the crawler reads must break it; flips of unread letters must not matter. An independent Python ant re-checks k <= 10.
The README gives the hand proof for all k >= 1 (seven phases, two loop invariants); these runs check it.

In [2]:
print((R / "p1_crawler.txt").read_text())

P1: explicit crawler C_k, k = 1..64, 16 periods each (registered in README)
main runs (minimal + 4 random-suffix rules per k, seed 20261025): 320
control flips of letters 0..2k+1 (must fail within one period): 4288
control flips of unread suffix letters (must be unchanged): 293
independent Python oracle comparisons (k <= 10): 140
failures: 0
VERDICT: P1 HELD



## Instrument oracle
The footprint instrument (`examples/square_ring_fillers_footprint.rs`) equals `fast::run` on final fingerprints, and an independent Python ant
reproduces its histograms and rule-switch outputs.

In [3]:
print("\n".join(l for l in (R / "oracle.txt").read_text().splitlines() if "Compiling" not in l and "Finished" not in l))

# commit d709628 dirty=0
ae619384fcf6cb6d28ca59f60930cc47c903546305352d6a1c6b23585a5ef092  target/release/examples/square_ring_fillers_footprint
c121329f5dbb8b59a5c8bf5f3aab66e3cf1cf147c6e8d22b6e2be92a3bb5684f  target/release/examples/rlrlll_macro_transition_resim
FINGERPRINT RLRRRL 1000000 identical (cdf341356679f163 251468)
FINGERPRINT RLRRRL 10000000 identical (eb14c6243ff5d2c9 2504980)
FINGERPRINT RRLRRRR 1000000 identical (d1e47648fb54e7a5 166998)
FINGERPRINT RRLRRRR 10000000 identical (1cac937f4eaf5be5 1668030)
FINGERPRINT RLLR 1000000 identical (7a5233bfa2bb4de7 26440)
FINGERPRINT RLLR 10000000 identical (48ebc08dbb2522ef 180932)
FINGERPRINT RRLLLRLLLRR 1000000 identical (35787bccd67efefe 18375)
FINGERPRINT RRLLLRLLLRR 10000000 identical (2e2020f19bd6a001 892007)
FINGERPRINT RLRR 1000000 identical (5f06b3ee33d3f191 17024)
FINGERPRINT RLRR 10000000 identical (49dd882c522be141 86561)
RL (512 board, window 65536, stop=escaped, steps=22381): OK
RLRRRL (512 board, window 65536, stop=

## P2/P3: exact revolution clock and footprint in all 25 census board fillers
Registered: `dt = 8(k+1) * swing + b_(r mod p)` with zero residual in all four directions (the `clock.py` criterion of the sweeper experiment),
and reads confined to counts 0..2k+2 with uniform shares on 0..2k+1.

In [4]:
txt = (R / "p23_analysis.txt").read_text().splitlines()
print("\n".join(txt[:2] + [l for l in txt if l.startswith("R") or "P2 HELD" in l or "FAILED" in l][:20] + txt[-3:]))

P2/P3 analysis: 25 census board fillers, blank 8192 board, 2^27 steps; window [2^26, 2^27]

RLRRRRLLR k=1 stop=budget visited=33573182 steps/visited=3.9978
   P2 HELD (p<=2: True); footprint: max read 4, max rel dev of shares 0..2k+1 = 0.00043, share of 2k+2 = 0.000051, new-cell rate x(2k+2) = 1.00033 -> P3 HELD
RLRRRRRLLL k=1 stop=budget visited=33573114 steps/visited=3.9978
   P2 HELD (p<=2: True); footprint: max read 4, max rel dev of shares 0..2k+1 = 0.00043, share of 2k+2 = 0.000051, new-cell rate x(2k+2) = 1.00033 -> P3 HELD
RLRRRRL k=1 stop=budget visited=33572936 steps/visited=3.9978
   P2 HELD (p<=2: True); footprint: max read 4, max rel dev of shares 0..2k+1 = 0.00043, share of 2k+2 = 0.000051, new-cell rate x(2k+2) = 1.00033 -> P3 HELD
RLRRRLLR k=1 stop=budget visited=33573221 steps/visited=3.9978
   P2 HELD (p<=2: True); footprint: max read 4, max rel dev of shares 0..2k+1 = 0.00043, share of 2k+2 = 0.000051, new-cell rate x(2k+2) = 1.00033 -> P3 HELD
RLRRRLLLLR k=1 stop=bu

## P4: board exit steps predicted to the step
Exact quadratics through the exits on 1024/2048/4096 boards (from the earlier board check) have leading coefficient exactly `2(k+1)` and
predict the exits on 8192, 16384 and 32768 boards and the census records exactly. The registered B1 failure of the board check was the
linear term.

In [5]:
print((R / "p4_predictions.txt").read_text()); print((R / "p4_analysis.txt").read_text())

P4 predictions: exact quadratic through exit steps on boards 1024, 2048, 4096
rule,k,alpha,beta,gamma,alpha==2(k+1),E8192,E16384,E32768
RLRRRRL,1,4,-49,1420,True,268035468,1072940428,4293363084
RRLRR,2,6,-49,773,True,402252549,1609810693,6440846085
RRRLRRRRR,3,8,-65,2363,True,536340795,2146421051,8587807035

P4: predicted vs observed exit steps (exact integers)
rule,G,predicted,observed,source,match
RLRRRRL,8192,268035468,268035468,sweep_lean rerun,EXACT
RRLRR,8192,402252549,402252549,sweep_lean rerun,EXACT
RRRLRRRRR,8192,536340795,536340795,sweep_lean rerun,EXACT
RLRRRRL,16384,1072940428,1072940428,sweep_lean rerun,EXACT
RRLRR,16384,1609810693,1609810693,sweep_lean rerun,EXACT
RRRLRRRRR,16384,2146421051,2146421051,sweep_lean rerun,EXACT
RLRRRRL,32768,4293363084,4293363084,sweep_lean rerun,EXACT
RLRRRRL,32768,4293363084,4293363084,census h1e10/chunk_57.csv,EXACT
RRLRR,32768,6440846085,6440846085,sweep_lean rerun,EXACT
RRLRR,32768,6440846085,6440846085,census h1e10/chunk_50.csv,EXACT
RR

![square-ring figure](results/fig_square_ring.png)

*(a) Visit counts of `RLRRRL` after 2e5 steps: a square of 4-visit cells, corner diagonals, the chaotic seed in the middle. (b) The north edge
of `RRLRRRR` (k = 2): rows of 2-pairs, (3, 5)-pairs and 6s. (c) Exit-step shortfall below `2(k+1)G^2`, fit on three boards, all later exits on
the lines. Generated by `scripts/figure.py`.*

## P5: the corner reads `T(2k+2)`; rules that never fill from blank run a transplanted ring
(a) Switching mid-fill to `RLRR`, `RRLRRR`, `RRRLRRRR` (same turns on counts 0..2k+2) leaves the visit field bit-identical for 1e8 steps, though
these rules do not fill from a blank board. (b) Flipping only `T(2k+2)` breaks the ring at the first corner.

In [6]:
print((R / "p5_analysis.txt").read_text())

P5: transplant of mid-fill states (8192 board, t0 + 1e8 steps)
k=1 t0=1000000: original RLRRRL#4 visited=25266200
  (a) RLRRRL>RLRR@1000000#4: identical position/heading/vhash/visited = True
  (b) RLRRRL>RLRRLL@1000000#4: first read > 4 at t0+2956; revolution at t0 = 8058 steps -> broke within one revolution = True; visited at end 763683 vs 25266200; last-window max read 64+, new cells 5780 vs 262227
k=2 t0=2000000: original RRLRRRR#6 visited=17012906
  (a) RRLRRRR>RRLRRR@2000000#6: identical position/heading/vhash/visited = True
  (b) RRLRRRR>RRLRRRL@2000000#6: first read > 6 at t0+1962; revolution at t0 = 13882 steps -> broke within one revolution = True; visited at end 871650 vs 17012906; last-window max read 64+, new cells 7875 vs 174831
k=3 t0=3000000: original RRRLRRRRR#8 visited=12885938
  (a) RRRLRRRRR>RRRLRRRR@3000000#8: identical position/heading/vhash/visited = True
  (b) RRRLRRRRR>RRRLRRRRL@3000000#8: first read > 8 at t0+3477; revolution at t0 = 19690 steps -> broke within

## P6: which census rules are rings? (Q15)
Every builder and every rule of length <= 12 satisfying `Pi(k)` was run for 2^27 steps on a blank 16384 board and the last 2^24 steps
classified by their read histogram: ring(k), two-visit (counts 0, 1 >= 90%), or other.

In [7]:
print((R / "p6_analysis.txt").read_text())

P6: census footprint classification (registered tests)
runs: 446 (escaped the 16384 board: 12)
classes: {'other': 206, 'ring(2)': 43, 'ring(1)': 159, 'two-visit': 20, 'ring(4)': 3, 'ring(5)': 4, 'ring(3)': 9, 'ring(7)': 1, 'ring(6)': 1}

(a) necessity: ring(k) rules 220, of which not Pi(k) for the same k: 2 ['RRLLLRLLLRLL', 'RRRLRRLLRRRR'] -> FAILED
(b) sufficiency: Pi rules 361, classified ring(k): 218 = 60.4% (registered band 30%-90%) -> HELD
(c) Q15: census builders 304: ring(k) 72.4% (>= 50% registered) -> HELD; two-visit 6.6% (10%-30% registered) -> FAILED; other 21.1%

Post hoc tables (not registered):
ring(k) count by k: [(1, 159), (2, 43), (3, 9), (4, 3), (5, 4), (6, 1), (7, 1)]
Pi rules by k: classified ring / total: {'1': (159, 256), '2': (42, 65), '3': (9, 18), '4': (3, 7), '5': (3, 5), '6': (1, 4), '7': (1, 3), '8': (0, 2), '9': (0, 1)}
Pi rules not ring(k), by (class, sweep stop at 1e8, census cls): [(('other', 'budget', 'blob'), 136), (('other', 'budget', 'log'), 2), (('o

In [8]:
rows = list(csv.DictReader(open(R / "p6_classes.csv")))
from collections import Counter
print("builders by class:", Counter(r["cls"] for r in rows if r["in_builders"] == "True"))
print("Pi rules by class:", Counter(r["cls"] for r in rows if r["pi_k"]))

builders by class: Counter({'ring(1)': 159, 'other': 64, 'ring(2)': 43, 'two-visit': 20, 'ring(3)': 9, 'ring(5)': 4, 'ring(4)': 3, 'ring(7)': 1, 'ring(6)': 1})
Pi rules by class: Counter({'ring(1)': 159, 'other': 143, 'ring(2)': 42, 'ring(3)': 9, 'ring(4)': 3, 'ring(5)': 3, 'ring(7)': 1, 'ring(6)': 1})


## Post hoc (not registered): the k = 0 member
`C_0` (one row of 1s over 2s) translates exactly when `T(0) = L, T(1) = R`; mirrored, that is every canonical rule starting `RL`. The
two-visit builder `RLLLRRRL` shows exactly this edge (a row of 1s over 2s, a corner diagonal of 3s).

In [9]:
print((R / "posthoc_k0.txt").read_text().splitlines()[-1])

translation iff T(0) = L and T(1) = R: 124 agree, 0 disagree


## What this shows, and what it does not
- **Shown (proof):** for every k >= 1 the straight-edge crawler exists exactly for turn tables `R^k L R^(k+1)` on counts 0..2k+1, moves 4 cells
  per `8(k+1)` steps and leaves every cell with `2(k+1)` visits. That is the `2(k+1)` law and the reason for the rule family.
- **Shown (exact, finite):** the 25 census board fillers run an exact two-phase revolution clock with slope `8(k+1)`; their board exits
  follow `2(k+1)G^2 + beta G + gamma` to the step; the corner needs `T(2k+2) = R`.
- **Not shown:** a proof that the corner and hence the ring persist forever (observed exact over hundreds of revolutions only); why some
  `Pi` rules reach the ring from a blank board and others do not (the seed reads letters the ring never reads); whether the octagonal spirals
  of the sweeper experiment are the k = 0 member (their edge is not checked here).
- **Next:** a corner lemma (finite trace per k at fixed corner phase) would close the ring as a macro-transition for all k; the onset question
  is the same kind of question as highway onset.